# pymfx — Advanced Visualization

This notebook covers three advanced visualization functions:

1. **`speed_heatmap`** — interactive map with trajectory segments coloured green → red by speed
2. **`compare_map`** — multiple flights overlaid on the same map with a colour legend
3. **`flight_3d`** — three-dimensional matplotlib plot (longitude / latitude / altitude)

**Requirements:**
```bash
pip install pymfx[viz]   # folium + matplotlib
```

In [ ]:
import sys, math, uuid
sys.path.insert(0, '..')  # if running from notebooks/

import matplotlib
matplotlib.use('Agg')   # use non-interactive backend for notebook rendering
import matplotlib.pyplot as plt

import pymfx
from pymfx import viz
from pymfx.models import (
    MfxFile, Meta, Trajectory, TrajectoryPoint, SchemaField,
    Events, Event, Index,
)
from pymfx import compute_checksum

print(f"pymfx {pymfx.__version__}")

## Build two demo missions

We create two flights over the same general area with slightly different paths and speeds — one faster, one slower.

In [ ]:
def make_mfx(
    drone_id: str,
    base_speed: float = 8.0,
    lat_offset: float = 0.0,
    date_start: str = '2026-05-10T08:00:00Z',
    date_end:   str = '2026-05-10T08:00:50Z',
    n: int = 50,
) -> MfxFile:
    BASE_LAT, BASE_LON = 43.2965 + lat_offset, 5.3698  # Marseille
    FREQ = 1.0

    raw_pts = [
        (
            round(float(i), 3),
            round(BASE_LAT + i * 0.00009, 7),
            round(BASE_LON + math.cos(i * 0.2) * 0.0003, 7),
            round(150.0 + math.sin(i * 0.25) * 15, 2),
            round(base_speed + math.sin(i * 0.3) * 2, 2),
            round((i * 7) % 360, 1),
        )
        for i in range(n)
    ]

    schema_fields = [
        SchemaField('t',        'float',   ['no_null']),
        SchemaField('lat',      'float',   ['no_null']),
        SchemaField('lon',      'float',   ['no_null']),
        SchemaField('alt_m',    'float32'),
        SchemaField('speed_ms', 'float32'),
        SchemaField('heading',  'float32'),
    ]
    points    = [TrajectoryPoint(t=r[0], lat=r[1], lon=r[2], alt_m=r[3], speed_ms=r[4], heading=r[5]) for r in raw_pts]
    raw_lines = [f"{r[0]:.3f} | {r[1]} | {r[2]} | {r[3]} | {r[4]} | {r[5]}" for r in raw_pts]

    ev_schema = [
        SchemaField('t',        'float', ['no_null']),
        SchemaField('type',     'str'),
        SchemaField('severity', 'str'),
        SchemaField('detail',   'str'),
    ]
    ev_list = [
        Event(t=0.0,  type='takeoff',  severity='info',    detail='ok'),
        Event(t=20.0, type='anomaly',  severity='warning', detail='wind'),
        Event(t=n-1,  type='landing',  severity='info',    detail='ok'),
    ]
    ev_raw = [f"{e.t:.3f} | {e.type} | {e.severity} | {e.detail}" for e in ev_list]

    lats = [p.lat for p in points];  lons = [p.lon for p in points]
    bbox = (round(min(lons), 7), round(min(lats), 7), round(max(lons), 7), round(max(lats), 7))

    return MfxFile(
        version='1.0', encoding='UTF-8',
        meta=Meta(
            id=f'uuid:{uuid.uuid4()}',
            drone_id=drone_id,
            drone_type='multirotor',
            pilot_id='FR-PILOT-0001',
            date_start=date_start,
            date_end=date_end,
            duration_s=n - 1,
            status='complete',
            application='inspection',
            location='Marseille, France',
            sensors=['rgb'],
            data_level='raw',
            license='CC-BY-4.0',
            contact='ops@inspect.fr',
        ),
        trajectory=Trajectory(
            frequency_hz=FREQ,
            schema_fields=schema_fields,
            points=points,
            checksum=compute_checksum(raw_lines),
            raw_lines=raw_lines,
        ),
        events=Events(
            schema_fields=ev_schema,
            events=ev_list,
            checksum=compute_checksum(ev_raw),
            raw_lines=ev_raw,
        ),
        index=Index(bbox=bbox, anomalies=1),
    )

mfx_fast = make_mfx('Mavic3-FAST-SN001', base_speed=12.0, lat_offset=0.000)
mfx_slow = make_mfx('Mavic3-SLOW-SN002', base_speed= 5.0, lat_offset=0.005)

print(pymfx.validate(mfx_fast))
print(pymfx.validate(mfx_slow))

## 1. Speed heatmap

`speed_heatmap()` draws each trajectory segment as a coloured `PolyLine`.
The colour scale (green = slow, red = fast) is computed from the actual speed range of the flight.

In [ ]:
m_heat = viz.speed_heatmap(mfx_fast)
m_heat  # renders inline in Jupyter

In [ ]:
# Show event markers as well
m_heat_ev = viz.speed_heatmap(mfx_fast, show_events=True)
m_heat_ev.save('speed_heatmap.html')
print("Saved to speed_heatmap.html")

In [ ]:
# Options: tile style, line weight
m_heat2 = viz.speed_heatmap(mfx_fast, tile='CartoDB positron', line_weight=6)
m_heat2

## 2. Compare map

`compare_map()` overlays multiple flights on the same map using distinct colours.
An auto-generated HTML legend identifies each flight by `drone_id` (or by a custom label).

In [ ]:
# Default labels = drone_id from each flight's meta
m_compare = viz.compare_map([mfx_fast, mfx_slow])
m_compare

In [ ]:
# Custom labels + show events
m_compare2 = viz.compare_map(
    [mfx_fast, mfx_slow],
    labels=['High-speed run', 'Low-speed run'],
    show_events=True,
)
m_compare2.save('compare_map.html')
print("Saved to compare_map.html")

In [ ]:
# Three flights at once
mfx_med = make_mfx('Mavic3-MED-SN003', base_speed=8.5, lat_offset=0.010)
m_three = viz.compare_map([mfx_fast, mfx_med, mfx_slow],
                          labels=['Fast', 'Medium', 'Slow'])
m_three

## 3. 3-D trajectory plot

`flight_3d()` uses `mpl_toolkits.mplot3d` to render a three-dimensional view of the flight path (longitude × latitude × altitude).

- `color_by='speed'` — each segment is coloured by speed (green = slow, red = fast)
- `azim` / `elev` — control the viewing angle
- Events appear as orange triangles

In [ ]:
fig = viz.flight_3d(mfx_fast)
plt.show()

In [ ]:
# Colour segments by speed
fig_spd = viz.flight_3d(mfx_fast, color_by='speed')
plt.show()

In [ ]:
# Show events + custom view angle
fig_ev = viz.flight_3d(mfx_fast, color_by='speed', show_events=True, azim=30, elev=35)
fig_ev.savefig('flight_3d.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to flight_3d.png")

In [ ]:
# Top-down view (looking straight down, elev=90)
fig_top = viz.flight_3d(mfx_fast, elev=89, azim=0, figsize=(8, 8))
fig_top.axes[0].set_title('Top-down view', fontsize=11)
plt.show()

## Summary

| Function | Returns | Description |
|---|---|---|
| `viz.speed_heatmap(mfx)` | `folium.Map` | Segments coloured green → red by speed |
| `viz.compare_map([mfx1, mfx2, ...])` | `folium.Map` | Multiple flights with legend |
| `viz.flight_3d(mfx, color_by='speed')` | `Figure` | 3-D lon / lat / alt matplotlib plot |

All three accept `show_events=True/False` to overlay event markers.